In [ ]:
from pathlib import Path
import json, hashlib, zipfile
ROOT=Path.cwd()
while not (ROOT/'pyproject.toml').exists():
    if ROOT==ROOT.parent: raise RuntimeError('Run inside repository')
    ROOT=ROOT.parent
D=ROOT/'docs/ntu-completion-20261011/stage4'
snapshot=json.loads((D/'結構化候選資料.json').read_text())
rows=json.loads((D/'all-school-review.json').read_text())
assert len(snapshot['schools'])==288 and len(rows)==4032
assert sum(len(s['observations']) for s in snapshot['schools'])==608
print({'schools':288,'field_rows':len(rows),'observations':608})


In [ ]:
from collections import Counter
quota=[r for r in rows if r['field']=='quota']
pending={r['portal_id'] for r in quota if not r['target_requirement_confirmed']}
ten={'quota','eligibility','language','documents','deadlines','calendar','housing','cost','courses','insurance_visa'}
core=[r for r in rows if r['portal_id'] in pending and r['field'] in ten]
assert len(pending)==282 and len(core)==2820
assert not any(r['target_requirement_confirmed'] for r in core)
print({'pending_schools':282,'ten_field_rows':2820,'fully_verified_rows':0,'readable_unread_rows':sum(bool(r['unread_candidate_body_count']) for r in rows)})


In [ ]:
manifest=json.loads((D/'reviewed-source-bodies-current-manifest.json').read_text())
archives={}
for part in manifest['archive_files']:
    path=ROOT/'sources/ntu'/part['archive_file']
    assert hashlib.sha256(path.read_bytes()).hexdigest()==part['sha256']
    archives[part['archive_file']]=zipfile.ZipFile(path)
for entry in manifest['entries']:
    body=archives[entry['archive_file']].read(entry['archive_body_path'])
    assert hashlib.sha256(body).hexdigest()==entry['sha256']
for archive_file in archives.values(): archive_file.close()
assert len(manifest['entries'])==478 and len(manifest['archive_files'])==16
print({'archived_original_bodies':478,'independent_zip_parts':16,'body_hashes_verified':True})
